<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS09/datasetPBL/Tugas2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load dan gabungkan dataset milikmu
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))
df = df.dropna(subset=['text', 'rating_review']).copy()

# 2. Bentuk kolom teks & label biner (1 = Positif [rating >= 4], 0 = Negatif [rating < 4])
df['Review_Text'] = df['name'].fillna('') + " " + df['text'].astype(str)
df['Labels'] = np.where(df['rating_review'] >= 4, 1, 0)

X = df['Review_Text'].values
y = df['Labels'].values

# Gabungkan stop_words Inggris + Bahasa Indonesia umum
stop_words_id = {
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'untuk', 'dengan',
    'pada', 'adalah', 'juga', 'saja', 'nya', 'ya', 'yg', 'ga', 'gak',
    'tidak', 'bisa', 'ada', 'atau', 'seperti', 'lebih', 'sangat', 'ke',
    'dalam', 'akan', 'oleh', 'karena', 'sudah', 'lagi', 'kalau', 'jika'
}
custom_stop_words = list(ENGLISH_STOP_WORDS.union(stop_words_id))

# Split data training (80%) dan testing (20%) yang identik untuk kedua model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50, stratify=y
)

print("Jumlah data train:", len(X_train), "| Jumlah data test:", len(X_test))
print("Distribusi kelas (0=Negatif, 1=Positif):", np.bincount(y))

Jumlah data train: 1784 | Jumlah data test: 446
Distribusi kelas (0=Negatif, 1=Positif): [ 244 1986]


In [2]:
# Ekstraksi fitur CountVectorizer dengan stop_words aktif
bow_sw = CountVectorizer(stop_words=custom_stop_words)
X_train_bow = bow_sw.fit_transform(X_train)
X_test_bow = bow_sw.transform(X_test)

# Latih model MultinomialNB
mnb_bow = MultinomialNB()
mnb_bow.fit(X_train_bow, y_train)

# Prediksi & Evaluasi
y_pred_train_bow = mnb_bow.predict(X_train_bow)
y_pred_test_bow = mnb_bow.predict(X_test_bow)

acc_train_bow = accuracy_score(y_train, y_pred_train_bow)
acc_test_bow = accuracy_score(y_test, y_pred_test_bow)

print("=== HASIL MODEL 1: MULTINOMIAL NB + COUNTVECTORIZER (STOP_WORDS) ===")
print(f"Jumlah Fitur Kosakata : {len(bow_sw.get_feature_names_out())}")
print(f"Akurasi Training      : {acc_train_bow:.4f}")
print(f"Akurasi Testing       : {acc_test_bow:.4f}")
print("\nConfusion Matrix (Testing):\n", confusion_matrix(y_test, y_pred_test_bow))
print("\nClassification Report (Testing):\n", classification_report(y_test, y_pred_test_bow, target_names=['Negatif (0)', 'Positif (1)']))

=== HASIL MODEL 1: MULTINOMIAL NB + COUNTVECTORIZER (STOP_WORDS) ===
Jumlah Fitur Kosakata : 4487
Akurasi Training      : 0.9641
Akurasi Testing       : 0.9350

Confusion Matrix (Testing):
 [[ 23  26]
 [  3 394]]

Classification Report (Testing):
               precision    recall  f1-score   support

 Negatif (0)       0.88      0.47      0.61        49
 Positif (1)       0.94      0.99      0.96       397

    accuracy                           0.93       446
   macro avg       0.91      0.73      0.79       446
weighted avg       0.93      0.93      0.93       446



In [3]:
# Ekstraksi fitur TfidfVectorizer dengan stop_words aktif
tfidf_sw = TfidfVectorizer(stop_words=custom_stop_words)
X_train_tfidf = tfidf_sw.fit_transform(X_train)
X_test_tfidf = tfidf_sw.transform(X_test)

# Latih model MultinomialNB
mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)

# Prediksi & Evaluasi
y_pred_train_tfidf = mnb_tfidf.predict(X_train_tfidf)
y_pred_test_tfidf = mnb_tfidf.predict(X_test_tfidf)

acc_train_tfidf = accuracy_score(y_train, y_pred_train_tfidf)
acc_test_tfidf = accuracy_score(y_test, y_pred_test_tfidf)

print("=== HASIL MODEL 2: MULTINOMIAL NB + TF-IDF (STOP_WORDS) ===")
print(f"Jumlah Fitur Kosakata : {len(tfidf_sw.get_feature_names_out())}")
print(f"Akurasi Training      : {acc_train_tfidf:.4f}")
print(f"Akurasi Testing       : {acc_test_tfidf:.4f}")
print("\nConfusion Matrix (Testing):\n", confusion_matrix(y_test, y_pred_test_tfidf))
print("\nClassification Report (Testing):\n", classification_report(y_test, y_pred_test_tfidf, target_names=['Negatif (0)', 'Positif (1)']))

# Tabel Perbandingan Langsung
df_compare = pd.DataFrame({
    'Ekstraksi Fitur': ['CountVectorizer + stop_words', 'TfidfVectorizer + stop_words'],
    'Akurasi Train': [round(acc_train_bow, 4), round(acc_train_tfidf, 4)],
    'Akurasi Test': [round(acc_test_bow, 4), round(acc_test_tfidf, 4)]
})
print("\n=== TABEL PERBANDINGAN HASIL ===")
print(df_compare.to_string(index=False))

=== HASIL MODEL 2: MULTINOMIAL NB + TF-IDF (STOP_WORDS) ===
Jumlah Fitur Kosakata : 4487
Akurasi Training      : 0.8997
Akurasi Testing       : 0.8924

Confusion Matrix (Testing):
 [[  1  48]
 [  0 397]]

Classification Report (Testing):
               precision    recall  f1-score   support

 Negatif (0)       1.00      0.02      0.04        49
 Positif (1)       0.89      1.00      0.94       397

    accuracy                           0.89       446
   macro avg       0.95      0.51      0.49       446
weighted avg       0.90      0.89      0.84       446


=== TABEL PERBANDINGAN HASIL ===
             Ekstraksi Fitur  Akurasi Train  Akurasi Test
CountVectorizer + stop_words         0.9641        0.9350
TfidfVectorizer + stop_words         0.8997        0.8924
